In [0]:
# Silver Layer: Orders Cleansing + Unified CDC (BATCH ITERATOR)

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lower, upper, to_timestamp,
    when, regexp_replace, coalesce, lit, current_timestamp,
    row_number, desc, try_to_date, substring_index
)
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE = "ecomstore.ecomlake.bronze_orders"
SILVER_TABLE = "ecomstore.ecomlake.silver_orders"
QUARANTINE_TABLE = "ecomstore.ecomlake.silver_quarantine_orders"
TEMP_STAGING_TABLE = "ecomstore.ecomlake.temp_orders_staging"

def clean_amount(c):
    """Strips symbols and nullifies negative amounts in one pass."""
    cleaned = regexp_replace(col(c), r"[^\d.]", "").cast(DoubleType())
    return when(cleaned < 0, None).otherwise(cleaned)

VALID_STATUSES = ["placed", "confirmed", "processing", "shipped", "out_for_delivery", "delivered", "cancelled", "returned"]

bronze_orders = spark.read.table(BRONZE_TABLE)

# 1. Fetch batches chronologically
batches_df = spark.sql(f"SELECT DISTINCT _batch_id FROM {BRONZE_TABLE} ORDER BY _batch_id").collect()
batches = [row['_batch_id'] for row in batches_df]

for current_batch in batches:
    print(f"\n🚀 Processing Batch: {current_batch}")
    
    incoming_batch = bronze_orders.filter(col("_batch_id") == current_batch)

    # 2. Deduplicate WITHIN the current batch
    window_spec = Window.partitionBy("order_id").orderBy(desc("updated_at"), desc("_ingestion_timestamp"))
    deduped_df = incoming_batch.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1).drop("row_num")

    # 3. Cleansing & Soft Delete Logic
    cleaned_df = (
        deduped_df
        .withColumn("order_id", upper(trim(col("order_id"))))
        .withColumn("customer_id", upper(trim(col("customer_id"))))
        .withColumn("status", lower(trim(col("status"))))
        .withColumn("status", when(col("status").isin(VALID_STATUSES), col("status")).otherwise(lit("unknown")))
        
        # Pre-calculate CDC flags
        .withColumn("_is_deleted", when(col("status") == "cancelled", lit(True)).otherwise(lit(False)))
        .withColumn("_deleted_at", when(col("status") == "cancelled", current_timestamp()).otherwise(lit(None).cast("timestamp")))

        .withColumn("order_date", coalesce(try_to_date(col("order_date"), "yyyy-MM-dd"), try_to_date(col("order_date"), "dd/MM/yyyy")))
        .withColumn("updated_at", coalesce(to_timestamp(col("updated_at")), to_timestamp(col("updated_at"), "yyyy-MM-dd HH:mm:ss")))
        .withColumn("created_at", coalesce(to_timestamp(col("created_at")), to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss")))

        .withColumn("total_amount", clean_amount("total_amount"))
        .withColumn("final_amount", clean_amount("final_amount"))
        .withColumn("discount_amount", when(col("discount_amount").isNotNull(), clean_amount("discount_amount")).otherwise(lit(0.0)))

        .withColumn("payment_method", lower(trim(col("payment_method"))))
        .withColumn("payment_method", when(col("payment_method").contains("credit"), lit("credit_card")).when(col("payment_method").contains("debit"), lit("debit_card")).otherwise(col("payment_method")))
        .withColumn("channel", lower(trim(col("channel"))))
        .withColumn("shipping_pincode", substring_index(trim(col("shipping_pincode")), ".", 1))
        .withColumn("shipping_pincode", when(col("shipping_pincode").rlike(r"^\d{6}$"), col("shipping_pincode").cast(IntegerType())).otherwise(None))
        .withColumn("_silver_processed_at", current_timestamp())
    )

    silver_schema_cols = [
        "order_id", "customer_id", "order_date", "status", "total_amount", 
        "discount_amount", "final_amount", "shipping_address_city", 
        "shipping_address_state", "shipping_pincode", "payment_method", 
        "channel", "created_at", "updated_at", "_batch_id", "_is_deleted", "_deleted_at", "_silver_processed_at"
    ]

    # Disk Staging (Breaks Lineage for memory safety)
    cleaned_df.select(*silver_schema_cols).coalesce(1).write.format("delta").mode("overwrite").saveAsTable(TEMP_STAGING_TABLE)
    staged_silver_df = spark.read.table(TEMP_STAGING_TABLE)

    # 4. Data Quality Gates
    good_records = staged_silver_df.filter(col("order_id").isNotNull() & col("customer_id").isNotNull() & col("order_date").isNotNull())
    bad_records = staged_silver_df.filter(col("order_id").isNull() | col("customer_id").isNull() | col("order_date").isNull())

    if bad_records.count() > 0:
        (
            bad_records
            .withColumn("rejection_reason",
                when(col("order_id").isNull(), "null_order_id")
                .when(col("customer_id").isNull(), "null_customer_id")
                .otherwise("invalid_or_null_order_date")
            )
            .coalesce(1)
            .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(QUARANTINE_TABLE)
        )

    # 5. CDC MERGE
    if spark.catalog.tableExists(SILVER_TABLE):
        silver_target = DeltaTable.forName(spark, SILVER_TABLE)
        (
            silver_target.alias("target")
            .merge(good_records.alias("source"), "target.order_id = source.order_id")
            .whenMatchedUpdate(
                condition="source.updated_at > target.updated_at",
                set={c: f"source.{c}" for c in silver_schema_cols if c not in ["order_id", "created_at"]}
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"✅ CDC MERGE complete for Batch: {current_batch}")
    else:
        good_records.coalesce(1).write.format("delta").mode("overwrite").option("delta.autoOptimize.optimizeWrite", "true").option("delta.autoOptimize.autoCompact", "true").saveAsTable(SILVER_TABLE)
        print(f"✅ Created initial table for Batch: {current_batch}")


# Clean up Disk Staging
spark.sql(f"DROP TABLE IF EXISTS {TEMP_STAGING_TABLE}")
